Ten skrypt Pythona wczytuje dokument PDF, przetwarza go i tworzy dwa rodzaje indeksów (podsumowujący i wektorowy) za pomocą biblioteki LlamaIndex. Następnie tworzy router zapytań, który automatycznie wybiera odpowiednie narzędzie (indeks podsumowujący lub wektorowy) do odpowiedzi na pytania użytkownika dotyczące treści dokumentu.

# Setup

In [ ]:
!uv pip install llama-index  llama-index-embeddings-huggingface

In [2]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

In [ ]:
import os
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter

from llama_index.llms.openai import OpenAI
from llama_index.core import Settings, VectorStoreIndex, SummaryIndex

from llama_index.core.tools import QueryEngineTool
from llama_index.core.query_engine.router_query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector
import nest_asyncio
from google.colab import userdata
from pprint import pp

nest_asyncio.apply()

*   `from llama_index.core import SimpleDirectoryReader`: Importuje klasę `SimpleDirectoryReader` do wczytywania dokumentów tekstowych z katalogu.
*   `from llama_index.core.node_parser import SentenceSplitter`: Importuje klasę `SentenceSplitter`, która dzieli teksty na zdania, co jest przydatne przy indeksowaniu.
*   `from llama_index.llms.openai import OpenAI`: **Importuje klasę `OpenAI`, która umożliwia korzystanie z modeli językowych udostępnianych przez OpenAI.** To główna różnica w porównaniu do poprzednich zestawów importów, które używały Groq lub Ollama.
*   `from llama_index.core import Settings, VectorStoreIndex`: Importuje klasy `Settings` i `VectorStoreIndex` do konfigurowania LlamaIndex i tworzenia indeksów wektorowych.
*   `from llama_index.embeddings.huggingface import HuggingFaceEmbedding`: Importuje klasę `HuggingFaceEmbedding`, która umożliwia generowanie osadzeń wektorowych za pomocą modeli z Hugging Face.
*   `from llama_index.core import SummaryIndex, VectorStoreIndex`: Ponownie importuje `VectorStoreIndex` oraz dodaje `SummaryIndex`.
*   `from llama_index.core.tools import QueryEngineTool`: Importuje klasę `QueryEngineTool`, która pozwala na opakowanie silnika zapytań w narzędzie.
*   `from llama_index.core.query_engine.router_query_engine import RouterQueryEngine`: Importuje klasę `RouterQueryEngine`, która umożliwia kierowanie zapytań do różnych silników zapytań.
*   `from llama_index.core.selectors import LLMSingleSelector`: Importuje klasę `LLMSingleSelector`, która wykorzystuje model językowy do wyboru najlepszego silnika zapytań.

In [ ]:
class CFG:
    model1 = "gpt-4o-mini"
    model2 = "BAAI/bge-small-en-v1.5"
    temperature = 0.1
    chunksize = 1024

In [ ]:
os.environ["OPENAI_API_KEY"] = userdata.get("openaivision")

llm = OpenAI(model=CFG.model1)

Settings.llm = llm

# Dane i model

In [ ]:
documents = SimpleDirectoryReader(input_files=["/content/agents_paper.pdf"]).load_data()

Ten kod wczytuje zawartość pliku PDF o nazwie "agents_paper.pdf" do pamięci programu, przygotowując go do dalszej analizy przez LlamaIndex.

*   `SimpleDirectoryReader(input_files=["metagpt.pdf"])`: Tworzy instancję klasy `SimpleDirectoryReader`, która jest częścią biblioteki LlamaIndex. Argument `input_files=["metagpt.pdf"]` określa listę plików, które mają zostać wczytane. W tym przypadku lista zawiera tylko jeden plik: "metagpt.pdf".
*   `.load_data()`: Wywołuje metodę `load_data()` na utworzonym obiekcie `SimpleDirectoryReader`. Ta metoda odczytuje zawartość plików określonych w argumencie `input_files` i zwraca listę obiektów `Document`, które reprezentują poszczególne dokumenty. Każdy obiekt `Document` zawiera tekst pliku oraz metadane, takie jak nazwa pliku.

Podsumowując, ten kod wczytuje zawartość pliku "metagpt.pdf" i tworzy z niej listę obiektów `Document`, które będą używane przez LlamaIndex do indeksowania i wyszukiwania informacji.


In [ ]:
Settings.embed_model = HuggingFaceEmbedding(model_name=CFG.model2)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Ten kod konfiguruje model osadzeń (embedding model), który będzie używany przez LlamaIndex do reprezentacji tekstu w formie wektorów numerycznych.

*   `Settings.embed_model = ...`: Ustawia globalny obiekt `embed_model` w LlamaIndex na nowo skonfigurowany model osadzeń.  Oznacza to, że wszystkie operacje wymagające generowania osadzeń będą domyślnie korzystać z tego modelu.
*   `HuggingFaceEmbedding(model_name = CFG.model2)`: Tworzy instancję klasy `HuggingFaceEmbedding`, która umożliwia korzystanie z modeli osadzeń dostępnych w bibliotece Hugging Face Transformers. Argument `model_name = CFG.model2` określa, który model ma być używany. Wartość `CFG.model2` to `"BAAI/bge-small-en-v1.5"`, więc program będzie korzystał z modelu BGE (Bidirectional Encoder Representations from Transformers) od BAAI, w wersji small i przeznaczonej dla języka angielskiego.

Podsumowując, ten kod konfiguruje LlamaIndex tak, aby używał modelu `BAAI/bge-small-en-v1.5` z Hugging Face do generowania osadzeń wektorowych tekstu. Osadzenia te będą wykorzystywane do reprezentacji dokumentów i zapytań w przestrzeni wektorowej, co umożliwi efektywne wyszukiwanie podobnych treści.

# Narzędzia

In [ ]:
splitter = SentenceSplitter(chunk_size=CFG.chunksize)

Ten kod tworzy instancję klasy `SentenceSplitter` i konfiguruje ją do dzielenia tekstu na fragmenty o określonym rozmiarze.

In [24]:
nodes = splitter.get_nodes_from_documents(documents)

Ten kod dzieli wczytane dokumenty na mniejsze fragmenty (nody) za pomocą skonfigurowanego rozdzielacza zdań (`SentenceSplitter`).

In [25]:
summary_index = SummaryIndex(nodes)

Ten kod tworzy indeks streszczeń (Summary Index) z przygotowanych fragmentów tekstu (nodów).

In [ ]:
vector_index = VectorStoreIndex(nodes)

Ten kod tworzy indeks wektorowy (Vector Store Index) z przygotowanych fragmentów tekstu (nodów).

In [27]:
summary_query_engine = summary_index.as_query_engine(
    response_mode="tree_summarize",
    use_async=True,
)

Ten kod tworzy silnik zapytań (Query Engine) dla indeksu streszczeń (`SummaryIndex`).

*   `summary_index.as_query_engine(...)`: Wywołuje metodę `as_query_engine()` na obiekcie `summary_index`. Ta metoda tworzy silnik zapytań, który może być używany do zadawania pytań dotyczących zawartości indeksu streszczeń.
*   `response_mode="tree_summarize"`: Określa tryb generowania odpowiedzi. `"tree_summarize"` oznacza, że silnik zapytań będzie rekurencyjnie streszczał fragmenty tekstu, aż do uzyskania ostatecznej odpowiedzi. Jest to przydatne dla długich dokumentów, ponieważ pozwala na efektywne syntezowanie informacji.
*   `use_async=True`: Włącza tryb asynchroniczny, co oznacza, że zapytania będą przetwarzane w tle, bez blokowania głównego wątku programu.


In [ ]:
vector_query_engine = vector_index.as_query_engine()

Ten kod tworzy silnik zapytań (Query Engine) dla indeksu wektorowego (`VectorStoreIndex`).

*   `vector_index.as_query_engine()`: Wywołuje metodę `as_query_engine()` na obiekcie `vector_index`. Ta metoda tworzy silnik zapytań, który może być używany do zadawania pytań dotyczących zawartości indeksu wektorowego.
*   `vector_query_engine = ...`: Przypisuje utworzony obiekt silnika zapytań do zmiennej `vector_query_engine`.


In [ ]:
summary_tool = QueryEngineTool.from_defaults(
    query_engine=summary_query_engine,
    description=("Useful for summarization questions related to FA paper"),
)

Ten kod tworzy narzędzie (Tool) oparte na silniku zapytań dla indeksu streszczeń.

In [ ]:
vector_tool = QueryEngineTool.from_defaults(
    query_engine=vector_query_engine,
    description=("Useful for retrieving specific context from the FA paper."),
)

Ten kod tworzy narzędzie (Tool) oparte na silniku zapytań dla indeksu wektorowego.


In [ ]:
query_engine = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=[
        summary_tool,
        vector_tool,
    ],
    verbose=True,
)

Ten kod tworzy silnik zapytań router (Router Query Engine), który automatycznie wybiera najlepsze narzędzie do odpowiedzi na zapytanie użytkownika.

# Test

In [32]:
response = query_engine.query("what is this document about?")
pp(str(response))

Selecting query engine 0: The question 'what is this document about?' relates to summarization, which aligns with the first choice that is useful for summarization questions..
('The document provides an in-depth exploration of advancements and challenges '
 'in the field of intelligent agents, particularly those utilizing large '
 'language models (LLMs). It covers various aspects such as the architecture '
 'of these agents, including their cognitive processes, memory, perception, '
 'reasoning, and action modules. The content emphasizes the integration of '
 'cognitive science principles into AI design, the evolution of agents through '
 'self-improvement, and the dynamics of multi-agent systems.\n'
 '\n'
 'Additionally, it addresses safety considerations, including intrinsic and '
 'extrinsic threats to these agents, and outlines vulnerabilities like '
 'jailbreak attacks and misalignment issues. The document also discusses the '
 "concept of superalignment to ensure agents' behavio

In [ ]:
response = query_engine.query("How do agents share information with other agents?")
pp(str(response))

Selecting query engine 1: The question asks for specific information about how agents share information, which aligns with retrieving specific context from the FA paper..
('Agents share information with other agents through various interaction '
 'types, including consensus-oriented interactions, collaborative learning, '
 'teaching or mentoring, and task-oriented collaborations. In '
 'consensus-oriented interactions, agents engage in multi-directional '
 'information flow to align goals and synthesize perspectives, leading to a '
 'shared understanding. Collaborative learning involves peer-to-peer '
 'information exchange, where agents share individual experiences to mutually '
 'improve their skills. Teaching or mentoring interactions are characterized '
 'by unidirectional information flow from an expert to a novice, focusing on '
 'knowledge and skill transfer. Task-oriented collaborations involve '
 'sequential or pipeline information flow, coordinating efforts to achieve '
 'sha

In [34]:
response = query_engine.query(
    "How do agents share information with other agents? Explain in detail"
)
pp(str(response))

Selecting query engine 1: The question asks for a detailed explanation of how agents share information, which requires retrieving specific context from the FA paper..
('Agents share information with other agents through various interaction '
 'mechanisms that facilitate communication and collaboration within a '
 'multi-agent system (MAS). These interactions can be categorized into '
 'different types, each serving specific purposes and influenced by the '
 "agents' roles, goals, and the context of their tasks.\n"
 '\n'
 '1. **Agent-Agent Interactions**: Agents engage in direct interactions with '
 'one another, which can take multiple forms. These interactions may involve '
 'negotiation, where agents discuss and reach agreements on shared objectives, '
 'or collaborative problem-solving, where they pool their knowledge to tackle '
 'complex tasks. The nature of these interactions can vary, including '
 'one-on-one dialogues or multi-agent discussions, depending on the complexity '
 '